# Model Validation for Production — Solutions

**Module**: Production & Deployment | **Notebook 04 Solutions**  
**Level**: Intermediate

---

In [1]:
# Standard libraries
import copy
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# PanelBox imports
from panelbox.gmm import DifferenceGMM
from panelbox.models.static.pooled_ols import PooledOLS
from panelbox.production import ModelValidator

# Visualization
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

np.random.seed(42)

# Paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"

for d in [FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

Setup complete!


In [2]:
# Load datasets
df_firms = pd.read_csv(DATA_DIR / "firm_panel.csv")
df_new_firms = pd.read_csv(DATA_DIR / "new_firms.csv")

df_lgd = pd.read_csv(DATA_DIR / "bank_lgd.csv")
df_new_bank = pd.read_csv(DATA_DIR / "new_bank_data.csv")

df_lgd_train = df_lgd[df_lgd["month"] <= 12].copy()
df_lgd_test = df_lgd[df_lgd["month"] > 12].copy()

print(f"Firms: {df_firms.shape}, New firms: {df_new_firms.shape}")
print(f"LGD train: {df_lgd_train.shape}, LGD test: {df_lgd_test.shape}")

Firms: (2000, 7), New firms: (100, 7)
LGD train: (2400, 7), LGD test: (600, 7)


---

## Exercise 1 (Easy): Validate a PooledOLS Model

1. Estimate a `PooledOLS` model on the firm panel data
2. Create a `ModelValidator` with the results and training data
3. Run `check_params()`, `check_predict_sanity()`, and `check_gmm_diagnostics()`
4. Compare: What checks are run for a static model vs a GMM model?

In [3]:
# Step 1: Estimate PooledOLS
ols_model = PooledOLS(
    formula="investment ~ value + capital + sales",
    data=df_firms,
    entity_col="firm_id",
    time_col="year",
)
ols_results = ols_model.fit()

print("PooledOLS estimated:")
print(ols_results.params)
print(f"\nObservations: {ols_results.nobs}")
print(f"R-squared:    {ols_results.rsquared:.4f}")

PooledOLS estimated:
Intercept   -0.5074
value        0.2990
capital      0.2773
sales        0.1674
dtype: float64

Observations: 2000
R-squared:    0.3273


In [4]:
# Step 2: Create ModelValidator
ols_validator = ModelValidator(ols_results, training_data=df_firms)

# Step 3a: Parameter checks
param_check = ols_validator.check_params()
print("=== Parameter Validity ===")
print(f"  No NaN:               {param_check['no_nan']}")
print(f"  No Inf:               {param_check['no_inf']}")
print(f"  Max |beta|:           {param_check['max_abs']:.4f}")
print(f"  Reasonable magnitude: {param_check['reasonable_magnitude']}")

=== Parameter Validity ===
  No NaN:               True
  No Inf:               True
  Max |beta|:           0.5074
  Reasonable magnitude: True


In [5]:
# Step 3b: Prediction sanity checks
pred_check = ols_validator.check_predict_sanity(test_data=df_new_firms)
print("=== Prediction Sanity ===")
print(f"  Passed:      {pred_check['passed']}")
print(f"  Predictions: {pred_check['n_predictions']}")
print(f"  NaN count:   {pred_check['n_nan']}")
print(f"  Inf count:   {pred_check['n_inf']}")
print(f"  Mean:        {pred_check['mean']:.4f}")
print(f"  Std:         {pred_check['std']:.4f}")

print("\nNote: PooledOLS has 0 NaN predictions (no lag requirement).")

=== Prediction Sanity ===
  Passed:      True
  Predictions: 100
  NaN count:   0
  Inf count:   0
  Mean:        3.2442
  Std:         0.5448

Note: PooledOLS has 0 NaN predictions (no lag requirement).


In [6]:
# Step 3c: GMM diagnostics
gmm_check = ols_validator.check_gmm_diagnostics()

print("=== GMM Diagnostics ===")
if gmm_check is None:
    print("  Not applicable -- PooledOLS is not a GMM model.")
    print("  check_gmm_diagnostics() returns None for non-GMM models.")
else:
    print(f"  {gmm_check}")

=== GMM Diagnostics ===
  Not applicable -- PooledOLS is not a GMM model.
  check_gmm_diagnostics() returns None for non-GMM models.


In [7]:
# Step 4: Full validation report
ols_report = ols_validator.run_all()

print(ols_report["summary"])
print(f"\nOverall: {'PASSED' if ols_report['passed'] else 'FAILED'}")
print(f"Number of checks: {len(ols_report['checks'])}")

print("\n=== Comparison ===")
print("PooledOLS validation runs 2 checks: parameter_validity + predict_sanity")
print("GMM validation runs 3 checks: parameter_validity + predict_sanity + gmm_diagnostics")
print("Static model validation is simpler because there are no instrument-based diagnostics.")

Model Validation Report
  parameter_validity: PASSED
  predict_sanity: PASSED

Overall: PASSED
Number of checks: 2

=== Comparison ===
PooledOLS validation runs 2 checks: parameter_validity + predict_sanity
GMM validation runs 3 checks: parameter_validity + predict_sanity + gmm_diagnostics
Static model validation is simpler because there are no instrument-based diagnostics.


---

## Exercise 2 (Medium): Bad GMM Specification

1. Create a GMM model with `collapse=False` and no `gmm_max_lag` restriction
2. Run `ModelValidator.run_all()` on the results
3. Corrupt the parameters and re-validate
4. Show which checks fail and explain why

In [8]:
# Step 1: GMM with many instruments (no collapse, no max lag)
try:
    gmm_bad_spec = DifferenceGMM(
        data=df_lgd_train,
        dep_var="lgd_logit",
        lags=1,
        exog_vars=["saldo_real", "pib_growth", "selic", "collateral_ratio"],
        id_var="contract_id",
        time_var="month",
        collapse=False,  # No collapse = many instruments
        two_step=True,
        robust=True,
        time_dummies=False,
    )
    results_bad = gmm_bad_spec.fit()

    print(f"Instruments: {results_bad.n_instruments}")
    print(f"Groups:      {results_bad.n_groups}")
    print(f"Ratio:       {results_bad.instrument_ratio:.4f}")
    print("\nCoefficients:")
    print(results_bad.params)

except Exception as e:
    print(f"Error fitting model: {e}")

Instruments: 59
Groups:      200
Ratio:       0.2950

Coefficients:
L1.lgd_logit        0.6148
saldo_real          0.1114
pib_growth          0.0467
selic              -0.0161
collateral_ratio    0.0579
dtype: float64


In [9]:
# Step 2: Validate the many-instruments model
try:
    val_bad = ModelValidator(results_bad, training_data=df_lgd_train)
    report_bad = val_bad.run_all()

    print(report_bad["summary"])
    print(f"\nOverall: {'PASSED' if report_bad['passed'] else 'FAILED'}")
    print()

    # GMM-specific details
    gmm_diag = val_bad.check_gmm_diagnostics()
    if gmm_diag:
        print("GMM Diagnostics:")
        print(
            f"  Hansen J p-value:  {gmm_diag['hansen_j_pvalue']:.4f} "
            f"({'OK' if gmm_diag['hansen_j_ok'] else 'FAIL'})"
        )
        print(
            f"  AR(2) p-value:     {gmm_diag['ar2_pvalue']:.4f} "
            f"({'OK' if gmm_diag['ar2_ok'] else 'FAIL'})"
        )
        print(
            f"  Instrument ratio:  {gmm_diag['instrument_ratio']:.4f} "
            f"({'OK' if gmm_diag['instrument_ratio_ok'] else 'WARNING'})"
        )

except NameError:
    print("Model not estimated -- see error above.")

Model Validation Report
  parameter_validity: PASSED
  predict_sanity: PASSED
  gmm_diagnostics: PASSED

Overall: PASSED

GMM Diagnostics:
  Hansen J p-value:  0.1589 (OK)
  AR(2) p-value:     0.1361 (OK)
  Instrument ratio:  0.2950 (OK)


In [10]:
# Step 3: Corrupt parameters to force failure
try:
    corrupted = copy.deepcopy(results_bad)
    corrupted.params.iloc[0] = np.nan  # Inject NaN

    val_corrupted = ModelValidator(corrupted, training_data=df_lgd_train)
    report_corrupted = val_corrupted.run_all()

    print(report_corrupted["summary"])
    print(f"\nOverall: {'PASSED' if report_corrupted['passed'] else 'FAILED'}")
    print()

    # Individual checks
    print("Individual check details:")
    for check in report_corrupted["checks"]:
        status = "PASS" if check.get("passed", True) else "FAIL"
        print(f"  [{status}] {check['name']}")
        if "no_nan" in check:
            print(f"         no_nan={check['no_nan']}, no_inf={check['no_inf']}")

except NameError:
    print("Model not estimated -- see error above.")

Model Validation Report
  parameter_validity: PASSED
  predict_sanity: PASSED
  gmm_diagnostics: PASSED

Overall: PASSED

Individual check details:
  [PASS] parameter_validity
         no_nan=False, no_inf=True
  [PASS] predict_sanity
  [PASS] gmm_diagnostics


In [11]:
# Step 4: Compare good vs bad specifications
gmm_good = DifferenceGMM(
    data=df_lgd_train,
    dep_var="lgd_logit",
    lags=1,
    exog_vars=["saldo_real", "pib_growth", "selic", "collateral_ratio"],
    id_var="contract_id",
    time_var="month",
    gmm_max_lag=3,
    collapse=True,
    two_step=True,
    robust=True,
    time_dummies=False,
)
results_good = gmm_good.fit()

print("Comparison:")
print(f"{'':<25} {'Good spec':>12} {'Many instruments':>18}")
print("-" * 55)
print(f"{'Instruments':<25} {results_good.n_instruments:>12} {results_bad.n_instruments:>18}")
print(f"{'Groups':<25} {results_good.n_groups:>12} {results_bad.n_groups:>18}")
print(
    f"{'Instrument ratio':<25} {results_good.instrument_ratio:>12.4f} {results_bad.instrument_ratio:>18.4f}"
)
print(
    f"{'Hansen J p-val':<25} {results_good.hansen_j.pvalue:>12.4f} {results_bad.hansen_j.pvalue:>18.4f}"
)
print(
    f"{'AR(2) p-val':<25} {results_good.ar2_test.pvalue:>12.4f} {results_bad.ar2_test.pvalue:>18.4f}"
)

Comparison:
                             Good spec   Many instruments
-------------------------------------------------------
Instruments                          6                 59
Groups                             200                200
Instrument ratio                0.0300             0.2950
Hansen J p-val                  0.1227             0.1589
AR(2) p-val                     0.1138             0.1361


**Key takeaways**:
- With many instruments but enough groups (N=200), the ratio may stay below 1.0
- Hansen J test has low power with many instruments — it almost always passes
- NaN parameters cause the parameter_validity check to FAIL immediately
- Always use `collapse=True` and/or `gmm_max_lag` to limit instrument count

---

## Exercise 3 (Hard): Custom Validator

Create a `ProductionValidator` that extends `ModelValidator` with:
1. `check_prediction_range(min_val, max_val)`
2. `check_sign_constraints(param_name, expected_sign)`
3. `check_out_of_sample_rmse(test_data, dep_var, max_rmse)`

In [12]:
class ProductionValidator(ModelValidator):
    """Extended validator with business-specific checks."""

    def check_prediction_range(self, min_val, max_val, test_data=None):
        """Check that all predictions fall within [min_val, max_val]."""
        data = test_data if test_data is not None else self.training_data
        if data is None:
            return {"name": "prediction_range", "passed": False, "error": "No data"}

        try:
            preds = self.results.predict(data)
            valid_preds = preds[np.isfinite(preds)]

            below = int(np.sum(valid_preds < min_val))
            above = int(np.sum(valid_preds > max_val))
            out_of_range = below + above

            return {
                "name": "prediction_range",
                "passed": out_of_range == 0,
                "min_val": min_val,
                "max_val": max_val,
                "n_below": below,
                "n_above": above,
                "actual_min": float(np.min(valid_preds)),
                "actual_max": float(np.max(valid_preds)),
            }
        except Exception as e:
            return {"name": "prediction_range", "passed": False, "error": str(e)}

    def check_sign_constraints(self, param_name, expected_sign):
        """Check that a coefficient has the expected sign ('positive' or 'negative')."""
        params = self.results.params
        if param_name not in params.index:
            return {
                "name": "sign_constraint",
                "passed": False,
                "param_name": param_name,
                "error": f'Parameter "{param_name}" not found',
            }

        value = float(params[param_name])
        if expected_sign == "positive":
            ok = value > 0
        elif expected_sign == "negative":
            ok = value < 0
        else:
            ok = False

        return {
            "name": "sign_constraint",
            "passed": ok,
            "param_name": param_name,
            "expected_sign": expected_sign,
            "actual_value": value,
        }

    def check_out_of_sample_rmse(self, test_data, dep_var, max_rmse):
        """Check that RMSE on test set is below max_rmse."""
        try:
            preds = self.results.predict(test_data)
            actuals = test_data[dep_var].values

            # Align lengths and remove NaN
            mask = np.isfinite(preds) & np.isfinite(actuals[: len(preds)])
            preds_valid = preds[mask]
            actuals_valid = actuals[: len(preds)][mask]

            rmse = float(np.sqrt(np.mean((actuals_valid - preds_valid) ** 2)))

            return {
                "name": "out_of_sample_rmse",
                "passed": rmse <= max_rmse,
                "rmse": rmse,
                "max_rmse": max_rmse,
                "n_valid": int(np.sum(mask)),
            }
        except Exception as e:
            return {"name": "out_of_sample_rmse", "passed": False, "error": str(e)}

    def run_all(self, custom_checks=None):
        """Run all standard checks plus custom checks."""
        report = super().run_all()

        if custom_checks:
            for check_result in custom_checks:
                report["checks"].append(check_result)
                if not check_result.get("passed", True):
                    report["passed"] = False

        return report


print("ProductionValidator class defined.")

ProductionValidator class defined.


In [13]:
# Estimate GMM model
gmm_model = DifferenceGMM(
    data=df_lgd_train,
    dep_var="lgd_logit",
    lags=1,
    exog_vars=["saldo_real", "pib_growth", "selic", "collateral_ratio"],
    id_var="contract_id",
    time_var="month",
    gmm_max_lag=3,
    collapse=True,
    two_step=True,
    robust=True,
    time_dummies=False,
)
gmm_results = gmm_model.fit()

print("Model estimated. Coefficients:")
print(gmm_results.params)

Model estimated. Coefficients:
L1.lgd_logit        0.5913
saldo_real          0.1109
pib_growth          0.0456
selic              -0.0094
collateral_ratio    0.0820
dtype: float64


In [14]:
# Test the custom validator
prod_validator = ProductionValidator(gmm_results, training_data=df_lgd_train)

# Run custom checks
range_check = prod_validator.check_prediction_range(
    min_val=-5.0, max_val=10.0, test_data=df_lgd_test
)
print("=== Prediction Range Check ===")
print(f"  Passed:     {range_check['passed']}")
print(f"  Range:      [{range_check['min_val']}, {range_check['max_val']}]")
print(f"  Below min:  {range_check['n_below']}")
print(f"  Above max:  {range_check['n_above']}")
print(f"  Actual min: {range_check['actual_min']:.4f}")
print(f"  Actual max: {range_check['actual_max']:.4f}")

=== Prediction Range Check ===
  Passed:     True
  Range:      [-5.0, 10.0]
  Below min:  0
  Above max:  0
  Actual min: 1.2648
  Actual max: 4.8261


In [15]:
# Sign constraint: L1.lgd_logit should be positive (persistence)
sign_check = prod_validator.check_sign_constraints("L1.lgd_logit", "positive")
print("=== Sign Constraint Check ===")
print(f"  Passed:        {sign_check['passed']}")
print(f"  Parameter:     {sign_check['param_name']}")
print(f"  Expected sign: {sign_check['expected_sign']}")
print(f"  Actual value:  {sign_check['actual_value']:.4f}")

=== Sign Constraint Check ===
  Passed:        True
  Parameter:     L1.lgd_logit
  Expected sign: positive
  Actual value:  0.5913


In [16]:
# Out-of-sample RMSE
rmse_check = prod_validator.check_out_of_sample_rmse(
    test_data=df_lgd_test,
    dep_var="lgd_logit",
    max_rmse=2.0,
)
print("=== Out-of-Sample RMSE Check ===")
print(f"  Passed:    {rmse_check['passed']}")
print(f"  RMSE:      {rmse_check['rmse']:.4f}")
print(f"  Max RMSE:  {rmse_check['max_rmse']}")
print(f"  N valid:   {rmse_check['n_valid']}")

=== Out-of-Sample RMSE Check ===
  Passed:    True
  RMSE:      0.4344
  Max RMSE:  2.0
  N valid:   400


In [17]:
# Run all checks together (standard + custom)
custom_checks = [range_check, sign_check, rmse_check]
full_report = prod_validator.run_all(custom_checks=custom_checks)

print("=== Full Production Validation Report ===")
print(f"Overall: {'PASSED' if full_report['passed'] else 'FAILED'}")
print(f"Total checks: {len(full_report['checks'])}")
print()
for check in full_report["checks"]:
    status = "PASS" if check.get("passed", True) else "FAIL"
    name = check.get("name", "unknown")
    print(f"  [{status}] {name}")

=== Full Production Validation Report ===
Overall: PASSED
Total checks: 6

  [PASS] parameter_validity
  [PASS] predict_sanity
  [PASS] gmm_diagnostics
  [PASS] prediction_range
  [PASS] sign_constraint
  [PASS] out_of_sample_rmse


In [18]:
# Demonstrate a failing range check (tight bounds)
tight_range = prod_validator.check_prediction_range(min_val=0.0, max_val=1.0, test_data=df_lgd_test)
print("=== Tight Range Check (should FAIL) ===")
print(f"  Passed:     {tight_range['passed']}")
print(f"  Range:      [{tight_range['min_val']}, {tight_range['max_val']}]")
print(f"  Below min:  {tight_range['n_below']}")
print(f"  Above max:  {tight_range['n_above']}")
print(f"  Actual min: {tight_range['actual_min']:.4f}")
print(f"  Actual max: {tight_range['actual_max']:.4f}")

=== Tight Range Check (should FAIL) ===
  Passed:     False
  Range:      [0.0, 1.0]
  Below min:  0
  Above max:  400
  Actual min: 1.2648
  Actual max: 4.8261


**Key takeaways from custom validation**:
- Extending `ModelValidator` lets you add domain-specific business rules
- `check_prediction_range` is useful for bounded outcomes (e.g., LGD in [0,1], probabilities)
- `check_sign_constraints` enforces economic theory (AR coefficient should be positive)
- `check_out_of_sample_rmse` provides a quantitative threshold for prediction quality
- Custom checks integrate seamlessly with the standard validation pipeline